New Notebook Created by Jupyter MCP Server

In [1]:
from pathlib import Path
import os, sys, inspect
repo = Path('/home/nahisaho/GitHub/jupytermind')
if str(repo / 'src') not in sys.path:
    sys.path.insert(0, str(repo / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(repo / 'benchmark/projects')
from ai_data_scientist import project_manager as pm, lifecycle, language_router
handle = pm.resolve_project('kaggle-v020-kaggle-exp-32-breast-cancer')
lifecycle.register_run('v020-exp-32', handle.notebook_path)
lifecycle.mark_write_start('v020-exp-32')
try:
    pm.ensure_notebook(handle)
    pm.ensure_data_dir(handle)
finally:
    lifecycle.mark_write_end('v020-exp-32')
print('日本語判定:', language_router.detect_language('良性・悪性を区別する特徴を日本語で分析してください'))
print('Notebook:', handle.notebook_path)
print('状態:', lifecycle.get_run_status('v020-exp-32'))

日本語判定: ja
Notebook: /home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-32-breast-cancer/notebooks/kaggle-v020-kaggle-exp-32-breast-cancer.ipynb
状態: RunStatus(run_id='v020-exp-32', state='running', active_cell_executions=0, pending_notebook_writes=0, locks_held=0, notebook_path='/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-32-breast-cancer/notebooks/kaggle-v020-kaggle-exp-32-breast-cancer.ipynb', reason=None)


In [2]:
import nbformat
setup_source = '''# 初期化とライフサイクル登録
from pathlib import Path
import os, sys, json, hashlib, io, zipfile, contextlib, warnings
from datetime import datetime, timezone
from dataclasses import asdict
repo = Path('/home/nahisaho/GitHub/jupytermind')
if str(repo / 'src') not in sys.path:
    sys.path.insert(0, str(repo / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(repo / 'benchmark/projects')
import numpy as np
import pandas as pd
import nbformat
from IPython.display import display, Markdown
from ai_data_scientist import project_manager as pm, lifecycle, language_router
from ai_data_scientist import ingestion, cleaning, eda, data_definition, analysis_assumptions
from ai_data_scientist import data_quality, sensitivity, stats_analysis, visualization, notebook_audit, insight_engine
RUN_ID = 'v020-exp-32'
handle = pm.resolve_project('kaggle-v020-kaggle-exp-32-breast-cancer')
lifecycle.register_run(RUN_ID, handle.notebook_path)
lifecycle.mark_write_start(RUN_ID)
try:
    pm.ensure_notebook(handle)
    data_dir = pm.ensure_data_dir(handle)
finally:
    lifecycle.mark_write_end(RUN_ID)
assert language_router.detect_language('良性・悪性の特徴を日本語で分析する') == 'ja'
phase_log = []
@contextlib.contextmanager
def phase(name):
    if lifecycle.is_cancel_requested(RUN_ID):
        raise RuntimeError('キャンセル要求により中止')
    lifecycle.mark_execution_start(RUN_ID)
    phase_log.append({'phase': name, 'event': 'start', 'utc': datetime.now(timezone.utc).isoformat()})
    try:
        yield
    except Exception as exc:
        lifecycle.mark_failed(RUN_ID)
        phase_log.append({'phase': name, 'event': 'failed', 'error_type': type(exc).__name__})
        raise
    finally:
        lifecycle.mark_execution_end(RUN_ID)
        phase_log.append({'phase': name, 'event': 'end', 'utc': datetime.now(timezone.utc).isoformat()})
display({'run_id': RUN_ID, '保存先': str(handle.notebook_path), '状態': asdict(lifecycle.get_run_status(RUN_ID))})
'''
acquisition_source = '''# Kaggle SDK検索・認証・取得と来歴
with phase('Kaggle取得'):
    SLUG = 'uciml/breast-cancer-wisconsin-data'
    acquisition_log = []
    csv_path = data_dir / 'data.csv'
    acquisition_route = 'Kaggle Python SDK'
    api_ok = False
    try:
        # 認証出力とSDKの内部出力を非表示にする。認証情報は読み出し・表示・保存しない。
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            from kaggle.api.kaggle_api_extended import KaggleApi
            api = KaggleApi()
            api.authenticate()
            found = api.dataset_list(search='breast cancer wisconsin')
            listing = api.dataset_list_files(SLUG)
        refs = [str(getattr(item, 'ref', '')) for item in found]
        names = [str(getattr(item, 'name', '')) for item in listing.files]
        acquisition_log.append({'検索語': 'breast cancer wisconsin', '検索結果slug': refs, '対象ファイル': names})
        if 'data.csv' not in names:
            raise FileNotFoundError('対象slugにdata.csvが存在しない')
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            api.dataset_download_file(SLUG, 'data.csv', path=str(data_dir), force=True, quiet=True)
        if not csv_path.exists():
            archived = data_dir / 'data.csv.zip'
            if not archived.exists():
                raise FileNotFoundError('SDK取得後にCSVまたはZIPが見つからない')
            with zipfile.ZipFile(archived) as z:
                if 'data.csv' not in z.namelist():
                    raise FileNotFoundError('ZIP内にdata.csvがない')
                csv_path.write_bytes(z.read('data.csv'))
        api_ok = True
    except Exception as exc:
        # SDKエラー文には認証情報が含まれる可能性があるため、型と状態コードだけを記録。
        acquisition_log.append({'段階': 'Kaggle API取得失敗', '例外型': type(exc).__name__, 'HTTP状態': getattr(exc, 'status', None)})
    if not api_ok:
        paper = Path('/home/nahisaho/kaggle/experiments/white-paper.md')
        if not paper.exists():
            display(acquisition_log)
            raise RuntimeError('取得失敗: 指定white-paper.mdがなく、許可された対応CSVを確認できない')
        import re
        paper_text = paper.read_text(encoding='utf-8')
        excerpts = [paper_text[max(0, m.start()-1500):m.end()+3000] for m in re.finditer(r'breast-cancer-wisconsin-data|Breast Cancer Wisconsin', paper_text, flags=re.I)]
        approved_urls = sorted(set(u.rstrip(').,;') for segment in excerpts for u in re.findall(r'https?://[^\\s<>\\"\\)]+', segment) if '.csv' in u.lower()))
        display({'取得失敗': acquisition_log, 'white-paperの候補URL': approved_urls})
        # 文脈と同一データを確認したURLだけを別セルで明示して使用する。自動推測しない。
        raise RuntimeError('取得失敗: white-paper候補URLの明示的な同一データ確認が必要')
    provenance = {'owner/dataset': SLUG, 'file': 'data.csv', 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(), 'sha256': hashlib.sha256(csv_path.read_bytes()).hexdigest(), 'route': acquisition_route, 'bytes': csv_path.stat().st_size, 'reference': 'https://www.kaggle.com/datasets/' + SLUG}
    (data_dir / 'provenance.json').write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding='utf-8')
    ingested = ingestion.ingest(ingestion.SourceSpec('csv', str(csv_path)), row_limit=100000)
    assert not ingested.truncated
    raw = ingested.dataframe
    display(provenance)
    display(acquisition_log)
    display({'行数': len(raw), '列数': len(raw.columns), '列名': raw.columns.tolist()})
'''
lifecycle.mark_write_start('v020-exp-32')
try:
    def initialize(nb):
        if nb.cells:
            raise RuntimeError('既存Notebookを上書きしない')
        nb.cells = [nbformat.v4.new_markdown_cell('# Breast Cancer Wisconsin：良性・悪性の説明可能な比較\n\nrun_id: `v020-exp-32`。公開の細胞核画像由来特徴を探索し、診断支援の検討材料を示す。個人の臨床診断、治療判断、安全な臨床運用の保証には使用しない。'), nbformat.v4.new_code_cell(setup_source), nbformat.v4.new_code_cell(acquisition_source)]
    pm.enqueue_write(handle, initialize)
finally:
    lifecycle.mark_write_end('v020-exp-32')
print('対象Notebookの初期セルを保存')

対象Notebookの初期セルを保存


In [5]:
definition_source = '''# データ定義manifest・クリーニング・意味的異常検査
with phase('データ定義・品質'):
    import urllib.request
    dictionary_url = 'https://archive.ics.uci.edu/static/public/17/breast+cancer+wisconsin+diagnostic.zip'
    try:
        with urllib.request.urlopen(dictionary_url, timeout=45) as response:
            dictionary_archive = response.read()
        with zipfile.ZipFile(io.BytesIO(dictionary_archive)) as z:
            dictionary_text = z.read('wdbc.names').decode('utf-8')
        (data_dir / 'wdbc.names').write_text(dictionary_text, encoding='utf-8')
        dictionary_status = 'verified'
        display({'定義照合元': dictionary_url, '辞書SHA256': hashlib.sha256(dictionary_text.encode()).hexdigest()})
        display(dictionary_text)
    except (OSError, KeyError, zipfile.BadZipFile) as exc:
        dictionary_status = 'inferred'
        display({'辞書取得失敗': type(exc).__name__, '扱い': '特徴定義は推定とし、物理単位を断定しない'})
    assert 'Unnamed: 32' in raw and raw['Unnamed: 32'].isna().all()
    usable = raw.drop(columns=['Unnamed: 32'])
    cleaned = cleaning.clean_dataset(usable, operation='drop_duplicates')
    df = cleaned.dataframe
    feature_names = [c for c in df if c not in ['id', 'diagnosis']]
    assert len(feature_names) == 30
    X = df[feature_names].copy()
    y = df['diagnosis'].map({'B': 0, 'M': 1})
    assert set(df['diagnosis']) == {'B', 'M'} and y.notna().all()
    quality_schema = {'id': {'not_null': True, 'unique': True}, 'diagnosis': {'not_null': True, 'allowed': ['B', 'M']}, **{c: {'min': 0, 'not_null': True} for c in feature_names}}
    assert set(quality_schema) == set(df.columns)
    anomalies = data_quality.detect_anomalies(df, schema=quality_schema)
    nonfinite_count = int((~np.isfinite(X.to_numpy())).sum())
    duplicate_feature_rows = int(X.duplicated().sum())
    display({'削除列': ['Unnamed: 32'], '全欠損セル数': len(raw), '重複削除行数': cleaned.rows_removed, '使用行数': len(df), '意味的異常': [asdict(a) for a in anomalies], '非有限値': nonfinite_count, '特徴完全重複行': duplicate_feature_rows})
    assert not anomalies and nonfinite_count == 0
    exploration = eda.explore(df)
    display(pd.DataFrame(exploration.missing_summary).T)
    display(pd.DataFrame(exploration.describe))
    counts = df['diagnosis'].value_counts().reindex(['B', 'M'])
    class_table = pd.DataFrame({'クラス': ['良性 B', '悪性 M'], '件数': counts.to_numpy(), '比率(%)': counts.to_numpy()/len(df)*100})
    display(class_table)
    definitions = {'radius': '重心から輪郭までの距離の平均', 'texture': 'グレースケール値の標準偏差', 'perimeter': '核の周長', 'area': '核の面積', 'smoothness': '半径長の局所変動', 'compactness': 'perimeter^2 / area - 1', 'concavity': '輪郭の凹部の程度', 'concave points': '輪郭の凹部の個数', 'symmetry': '対称性', 'fractal_dimension': 'coastline approximation - 1'}
    summaries = {'mean': '画像内の細胞核特徴の平均', 'se': '画像内の細胞核特徴の標準誤差', 'worst': '画像内で最も大きい3値の平均（患者の最悪予後ではない）'}
    FV = data_definition.FieldValue
    variables = {'id': {'definition': FV('識別子。予測に不使用', 'verified', 'CSV列および辞書'), 'patient_linkage': FV(None, 'unknown')}, 'diagnosis': {'definition': FV('B=良性、M=悪性', dictionary_status, dictionary_url), 'gold_standard_details': FV(None, 'unknown')}}
    for c in feature_names:
        base, suffix = c.rsplit('_', 1)
        variables[c] = {'definition': FV(definitions[base], dictionary_status, dictionary_url), 'aggregation': FV(summaries[suffix], dictionary_status, dictionary_url), 'unit': FV(None, 'unknown', '辞書に物理単位・校正尺度の明記なし')}
    definition_manifest = data_definition.build_manifest(source={'slug': FV(SLUG, 'verified', provenance['reference']), 'file_sha256': FV(provenance['sha256'], 'verified'), 'dictionary': FV(dictionary_url, dictionary_status)}, dataset_scope={'measurement_pathway': FV('乳房腫瘤の細針吸引(FNA)デジタル画像の細胞核', dictionary_status, dictionary_url), 'sampling': FV('公開診断データ。一般集団の有病率を表す標本ではない', dictionary_status, dictionary_url), 'patient_independence': FV(None, 'unknown'), 'demographics': FV(None, 'unknown')}, variables=variables, transformations=('全欠損列Unnamed: 32削除', '完全重複削除', 'idを予測・相関から除外', '悪性を陽性1として符号化'))
    unresolved = definition_manifest.unresolved_fields()
    inferred = [(f'source.{k}', v) for k,v in definition_manifest.source.items() if v.status == 'inferred'] + [(f'variables.{c}.{k}', v) for c,fields in variables.items() for k,v in fields.items() if v.status == 'inferred']
    display({'未解決定義': [(name, asdict(value)) for name, value in unresolved], '推定定義': [(name, asdict(value)) for name, value in inferred]})
    (data_dir / 'data_definition_manifest.json').write_text(json.dumps(asdict(definition_manifest), ensure_ascii=False, indent=2), encoding='utf-8')
    assumption_manifest = analysis_assumptions.AnalysisAssumptionManifest(analysis_scope={'対象': 'この公開標本内の記述・関連と内部予測性能', '陽性': '悪性 M', '臨床適用': '未検証'}, assumptions=(analysis_assumptions.Assumption('sampling-independent', '各行が独立な患者・症例である', 'assumed', impact_if_false='内部検証で患者漏洩、CI過小評価', conclusion_critical=True), analysis_assumptions.Assumption('no-leakage', '標準化・モデル学習を訓練foldだけで行う', 'tested', impact_if_false='内部予測性能が過大になる'), analysis_assumptions.Assumption('transfer', '施設・装置・対象集団が変わっても同じ性能', 'rejected', impact_if_false='臨床適用できない', conclusion_critical=True), analysis_assumptions.Assumption('clinical-cost', '見逃しと過剰判定の具体的効用は未取得', 'assumed', impact_if_false='最適な臨床閾値は決定できない', conclusion_critical=True)), causal_scope='associational', sampling={'n': len(df), 'seed': 32, 'design': '既存公開標本、層化内部検証'})
    assumption_findings = analysis_assumptions.check_manifest(assumption_manifest)
    display({'分析前提': asdict(assumption_manifest), '全指摘': [asdict(f) for f in assumption_findings]})
    (data_dir / 'analysis_assumptions_manifest.json').write_text(json.dumps(asdict(assumption_manifest), ensure_ascii=False, indent=2), encoding='utf-8')
'''
comparison_source = '''# 特徴の説明可能な比較と強相関
with phase('特徴比較・相関'):
    from scipy.stats import mannwhitneyu, rankdata
    rows = []
    for c in feature_names:
        b, m = X.loc[y==0, c], X.loc[y==1, c]
        pooled_sd = np.sqrt(((len(b)-1)*b.var(ddof=1)+(len(m)-1)*m.var(ddof=1))/(len(b)+len(m)-2))
        hedges_g = ((m.mean()-b.mean())/pooled_sd)*(1-3/(4*(len(b)+len(m))-9))
        u, p = mannwhitneyu(m, b, alternative='two-sided')
        auc_single = u/(len(b)*len(m))
        rows.append({'特徴': c, '良性中央値': b.median(), '良性Q1': b.quantile(.25), '良性Q3': b.quantile(.75), '悪性中央値': m.median(), '悪性Q1': m.quantile(.25), '悪性Q3': m.quantile(.75), 'Hedges_g(M-B)': hedges_g, '単変量AUC': auc_single, 'p': p})
    feature_table = pd.DataFrame(rows)
    pvals = feature_table['p'].to_numpy()
    order = np.argsort(pvals)
    adjusted = np.minimum.accumulate((pvals[order]*len(pvals)/np.arange(1,len(pvals)+1))[::-1])[::-1]
    feature_table.loc[order, 'BH_q'] = np.minimum(adjusted, 1)
    feature_table['効果量絶対値'] = feature_table['Hedges_g(M-B)'].abs()
    feature_table = feature_table.sort_values('効果量絶対値', ascending=False).reset_index(drop=True)
    display(feature_table.round(6))
    pearson, spearman = X.corr(method='pearson'), X.corr(method='spearman')
    pairs = []
    for i,a in enumerate(feature_names):
        for b in feature_names[i+1:]:
            if abs(pearson.loc[a,b]) >= .90:
                pairs.append({'特徴1': a, '特徴2': b, 'Pearson_r': pearson.loc[a,b], 'Spearman_r': spearman.loc[a,b], '良性内_r': X.loc[y==0,[a,b]].corr().iloc[0,1], '悪性内_r': X.loc[y==1,[a,b]].corr().iloc[0,1]})
    strong_pairs = pd.DataFrame(pairs).sort_values('Pearson_r', key=abs, ascending=False)
    display({'強相関基準': '|Pearson r| >= 0.90', '該当ペア数': len(strong_pairs), '全候補ペア数': len(feature_names)*(len(feature_names)-1)//2})
    display(strong_pairs.round(6))
    correlation_result = stats_analysis.correlation(df, 'radius_mean', 'perimeter_mean', language='ja')
    display(asdict(correlation_result))
    display(Markdown('相関の解釈：'+correlation_result.interpretation+' 半径・周長・面積の幾何学的な冗長性と、クラス混合による関連を区別する。相関は因果効果でも独立な寄与でもない。'))
    feature_table.to_csv(data_dir / 'feature_comparison.csv', index=False)
    strong_pairs.to_csv(data_dir / 'strong_correlations.csv', index=False)
'''
model_source = '''# 内部検証：単純ルールと説明可能なロジスティック回帰
with phase('内部予測検証'):
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier, export_text
    from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score, brier_score_loss
    reduced_features = ['radius_mean', 'texture_mean', 'concave points_mean', 'smoothness_mean']
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=32)
    folds = list(cv.split(X,y))
    model_specs = {'多数クラス(常に良性)': None, '1段決定木(30特徴)': DecisionTreeClassifier(max_depth=1, random_state=32), '4特徴ロジスティック': make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=3000, random_state=32)), '30特徴L2ロジスティック': make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=3000, random_state=32))}
    oof = {}
    model_rows = []
    def metrics_at(y_true, p, threshold=.5):
        tn, fp, fn, tp = confusion_matrix(y_true, np.asarray(p)>=threshold, labels=[0,1]).ravel()
        return {'閾値': threshold, 'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp), '感度': tp/(tp+fn), '特異度': tn/(tn+fp), '適合率': tp/(tp+fp) if tp+fp else np.nan, '陰性的中率': tn/(tn+fn) if tn+fn else np.nan, '正解率': (tp+tn)/len(y_true), 'balanced_accuracy': .5*(tp/(tp+fn)+tn/(tn+fp))}
    for name, estimator in model_specs.items():
        use = reduced_features if name=='4特徴ロジスティック' else feature_names
        p = np.zeros(len(y)) if estimator is None else cross_val_predict(estimator, X[use], y, cv=folds, method='predict_proba', n_jobs=1)[:,1]
        oof[name] = p
        model_rows.append({'モデル': name, **metrics_at(y,p), 'ROC_AUC': roc_auc_score(y,p), 'PR_AUC(AP)': average_precision_score(y,p), 'Brier': brier_score_loss(y,p)})
    model_table = pd.DataFrame(model_rows)
    display(model_table.round(5))
    threshold_table = pd.DataFrame([metrics_at(y,oof['30特徴L2ロジスティック'], t) for t in [.1,.2,.3,.5,.7]])
    display(Markdown('悪性を陽性。FN=悪性を良性と判定する見逃し、FP=良性を悪性と判定する過剰判定。閾値は臨床的に最適化したものではなく、事前指定の内部感度分析。'))
    display(threshold_table.round(5))
    def wilson(k,n):
        z=1.959963984540054
        center=(k/n+z*z/(2*n))/(1+z*z/n)
        half=z*np.sqrt((k/n)*(1-k/n)/n+z*z/(4*n*n))/(1+z*z/n)
        return center-half, center+half
    ci_rows=[]
    for name,p in oof.items():
        vals=metrics_at(y,p)
        ci_rows.append({'モデル':name, '感度Wilson95%': wilson(vals['TP'],vals['TP']+vals['FN']), '特異度Wilson95%':wilson(vals['TN'],vals['TN']+vals['FP'])})
    display(pd.DataFrame(ci_rows))
    display(Markdown('Wilson区間は症例独立の二項近似であり、fold間の学習依存性や施設間の変動を含まない。OOFは各症例を学習に使わずに予測するが、外部検証ではない。'))
    fitted_lr = make_pipeline(StandardScaler(), LogisticRegression(C=1,max_iter=3000,random_state=32)).fit(X,y)
    coefficients = pd.DataFrame({'特徴': feature_names, '標準化係数': fitted_lr[-1].coef_[0], '1標準偏差当たりOR': np.exp(fitted_lr[-1].coef_[0])}).sort_values('標準化係数', key=abs, ascending=False)
    display(coefficients.round(5))
    stump=DecisionTreeClassifier(max_depth=1,random_state=32).fit(X,y)
    display({'全標本の記述ルール(検証foldのルールではない)': export_text(stump, feature_names=feature_names)})
    model_table.to_csv(data_dir / 'model_comparison.csv',index=False)
    threshold_table.to_csv(data_dir / 'threshold_risks.csv',index=False)
'''
chart_source = '''# 日本語図表：クラス比率、分布、相関、誤分類、係数
with phase('図表作成'):
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    # 公開render_chartで日本語フォントを設定してから、複合図を描画する。
    class_png = visualization.render_chart(class_table, kind='bar', x='クラス', y='件数', title='良性・悪性のクラス構成', xlabel='診断ラベル', ylabel='症例数')
    display(visualization.build_image_output(class_png)['data'], raw=True)
    fig,axes=plt.subplots(2,3,figsize=(16,9))
    top_features=feature_table.head(6)['特徴'].tolist()
    for ax,c in zip(axes.flat,top_features):
        ax.boxplot([X.loc[y==0,c],X.loc[y==1,c]], tick_labels=['良性 B','悪性 M'], showfliers=True)
        ax.set_title(c); ax.set_xlabel('診断ラベル'); ax.set_ylabel('特徴値（物理単位未確認）')
    fig.suptitle('効果量上位6特徴の分布：四分位範囲と外れ値'); fig.tight_layout()
    display(fig); plt.close(fig)
    selected=['radius_mean','perimeter_mean','area_mean','concavity_mean','concave points_mean','radius_worst','perimeter_worst','area_worst','concave points_worst','texture_mean']
    fig,ax=plt.subplots(figsize=(12,10))
    im=ax.imshow(pearson.loc[selected,selected],vmin=-1,vmax=1,cmap='coolwarm')
    ax.set_xticks(range(len(selected)),selected,rotation=65,ha='right'); ax.set_yticks(range(len(selected)),selected)
    for i in range(len(selected)):
        for j in range(len(selected)):
            ax.text(j,i,f'{pearson.loc[selected[i],selected[j]]:.2f}',ha='center',va='center',fontsize=8)
    fig.colorbar(im,ax=ax,label='Pearson相関係数 r')
    ax.set_title('サイズ・形状特徴の強相関（全標本）'); ax.set_xlabel('特徴'); ax.set_ylabel('特徴'); fig.tight_layout()
    display(fig); plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    for ax,t in zip(axes,[.5,.2]):
        cm=confusion_matrix(y,oof['30特徴L2ロジスティック']>=t,labels=[0,1])
        ax.imshow(cm,cmap='Blues')
        for i in range(2):
            for j in range(2): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=18,color='black')
        ax.set_xticks([0,1],['良性','悪性']);ax.set_yticks([0,1],['良性','悪性'])
        ax.set_xlabel('OOF予測');ax.set_ylabel('実際のラベル');ax.set_title(f'30特徴ロジスティック：閾値 {t}')
    fig.suptitle('混同行列：下左が見逃し(FN)、上右が過剰判定(FP)');fig.tight_layout();display(fig);plt.close(fig)
    plot_coeff=coefficients.head(12).iloc[::-1]
    fig,ax=plt.subplots(figsize=(10,7));ax.barh(plot_coeff['特徴'],plot_coeff['標準化係数']);ax.axvline(0,color='black',linewidth=.7)
    ax.set_title('L2ロジスティック係数：独立な因果効果ではない');ax.set_xlabel('1標準偏差当たりlog odds（悪性）');ax.set_ylabel('特徴');fig.tight_layout();display(fig);plt.close(fig)
    font_path=font_manager.findfont(plt.rcParams['font.family'])
    display({'図表数':5, '日本語フォント':font_path, '注意':'1セル内の各PNGについて最終セルでラベル・日本語字形を検査しvisual_audit=Trueを適用する'})
'''
lifecycle.mark_write_start(RUN_ID)
try:
    def append_analysis(nb):
        for source in [definition_source, comparison_source, model_source, chart_source]:
            nb.cells.append(nbformat.v4.new_code_cell(source))
    pm.enqueue_write(handle, append_analysis)
finally:
    lifecycle.mark_write_end(RUN_ID)
print('定義・比較・内部検証・図表の4セルを追加')

定義・比較・内部検証・図表の4セルを追加


In [13]:
request1 = '初回分析ではサイズ・凹み特徴の差と30特徴モデルの高い内部性能が見られましたが、強い共線性と1回の分割による不確実性が残ります。層化分割の乱数、L2正則化、4特徴への縮約を変えて感度と見逃し件数を比較し、主要特徴の差が外れ値処理でも維持されるか、回帰係数の符号がfold間で安定するかを確認してください。'
iteration1_source = '''# 反復1：分割・正則化・縮約・外れ値の感度分析
with phase('反復1・感度分析'):
    sensitivity_rows=[]
    coefficient_samples=[]
    def evaluate_spec(seed, C, features):
        selected_features=feature_names if features=='30特徴' else reduced_features
        spec_folds=list(StratifiedKFold(5,shuffle=True,random_state=seed).split(X,y))
        estimator=make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=3000,random_state=seed))
        p=cross_val_predict(estimator,X[selected_features],y,cv=spec_folds,method='predict_proba',n_jobs=1)[:,1]
        scores=metrics_at(y,p)
        sensitivity_rows.append({'seed':seed,'C':C,'features':features,**scores,'ROC_AUC':roc_auc_score(y,p)})
        if C==1 and features=='30特徴':
            for fold_id,(train,test) in enumerate(spec_folds):
                fitted=make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=3000,random_state=seed)).fit(X.iloc[train],y.iloc[train])
                coefficient_samples.append({'seed':seed,'fold':fold_id,**dict(zip(feature_names,fitted[-1].coef_[0]))})
        return scores['感度']
    sensitivity_plan=sensitivity.SensitivityPlan({'seed':[32,7], 'C':[1,.1,10], 'features':['30特徴','4特徴']},max_runs=12)
    sensitivity_report=sensitivity.run_sensitivity(sensitivity_plan,evaluate_spec,stability_tolerance=.05)
    sensitivity_table=pd.DataFrame(sensitivity_rows)
    display(sensitivity_table.round(5))
    display({'感度のstability_tolerance':.05,'stable':sensitivity_report.stable,'max_relative_deviation':sensitivity_report.max_relative_deviation,'注意':'同じ569症例の再利用。12仕様は独立な12研究ではない。stableは感度だけの相対変動判定。'})
    coefficient_frame=pd.DataFrame(coefficient_samples)[feature_names]
    coefficient_stability=pd.DataFrame({'特徴':feature_names,'fold係数最小':coefficient_frame.min().to_numpy(),'fold係数最大':coefficient_frame.max().to_numpy(),'正係数fold比率':(coefficient_frame>0).mean().to_numpy()})
    display(coefficient_stability.round(5))
    effect_rows=[]
    for c in feature_table.head(6)['特徴']:
        for treatment in ['raw','winsor_1_99']:
            series=X[c] if treatment=='raw' else X[c].clip(X[c].quantile(.01),X[c].quantile(.99))
            b,m=series[y==0],series[y==1]
            sd=np.sqrt(((len(b)-1)*b.var(ddof=1)+(len(m)-1)*m.var(ddof=1))/(len(y)-2))
            effect_rows.append({'特徴':c,'外れ値扱い':treatment,'Hedges_g(M-B)':(m.mean()-b.mean())/sd*(1-3/(4*len(y)-9))})
    effect_sensitivity=pd.DataFrame(effect_rows)
    display(effect_sensitivity.round(5))
    display(Markdown('winsor化は全標本の記述的比較だけに適用し、モデル性能推定には使わない。極端値は誤りと断定せず、主解析では削除しない。'))
    sensitivity_table.to_csv(data_dir/'sensitivity_results.csv',index=False)
    coefficient_stability.to_csv(data_dir/'coefficient_stability.csv',index=False)
    (data_dir/'sensitivity_report.json').write_text(json.dumps(asdict(sensitivity_report),ensure_ascii=False,indent=2),encoding='utf-8')
'''
chart_helper = '''    from matplotlib.ft2font import FT2Font
    from matplotlib.text import Text
    chart_checks=[]
    def emit_figure(fig, description, legend):
        fig.canvas.draw()
        font_file=font_manager.findfont(font_manager.FontProperties(family=plt.rcParams['font.family']))
        charmap=FT2Font(font_file).get_charmap()
        texts=[obj.get_text() for obj in fig.findobj(Text) if obj.get_text()]
        missing=sorted(set(ord(ch) for text in texts for ch in text if ord(ch)>127 and ord(ch) not in charmap))
        check={'title':description,'xlabel':[ax.get_xlabel() for ax in fig.axes if ax.get_xlabel()],'ylabel':[ax.get_ylabel() for ax in fig.axes if ax.get_ylabel()],'legend':legend,'missing_glyphs':missing,'font':font_file}
        if missing:
            raise RuntimeError('日本語字形不足: '+str(missing))
        chart_checks.append(check)
        buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=110,bbox_inches='tight')
        payload=buffer.getvalue()
        (handle.root/'figures').mkdir(exist_ok=True)
        (handle.root/'figures'/f'chart-{len(chart_checks)}.png').write_bytes(payload)
        display(visualization.build_image_output(payload)['data'],raw=True)
        plt.close(fig)
'''
chart_source = chart_source.replace("    # 公開render_chart", chart_helper+"    # 公開render_chart")
chart_source = chart_source.replace("    display(visualization.build_image_output(class_png)['data'], raw=True)", "    class_font=font_manager.findfont(font_manager.FontProperties(family=plt.rcParams['font.family']))\n    from matplotlib.ft2font import FT2Font\n    chars=FT2Font(class_font).get_charmap()\n    class_missing=sorted({ord(c) for c in '良性悪性のクラス構成診断ラベル症例数件' if ord(c)>127 and ord(c) not in chars})\n    assert not class_missing\n    chart_checks.append({'title':'良性・悪性のクラス構成','xlabel':['診断ラベル'],'ylabel':['症例数'],'legend':'件数','missing_glyphs':class_missing,'font':class_font})\n    (handle.root/'figures').mkdir(exist_ok=True)\n    (handle.root/'figures'/'chart-1.png').write_bytes(class_png)\n    display(visualization.build_image_output(class_png)['data'], raw=True)")
chart_source = chart_source.replace('display(fig); plt.close(fig)', "emit_figure(fig, ax.get_title() if 'ax' in locals() else '特徴分布', '良性 B／悪性 M、箱=IQR、線=中央値、点=外れ値')")
chart_source = chart_source.replace('display(fig);plt.close(fig)', "emit_figure(fig, fig._suptitle.get_text() if fig._suptitle else ax.get_title(), '色=相関または件数、棒=係数。カテゴリは軸に明示')")
chart_source = chart_source.replace("font_manager.findfont(plt.rcParams['font.family'])", "font_manager.findfont(font_manager.FontProperties(family=plt.rcParams['font.family']))")
chart_source += "\n    display({'chart_checks':chart_checks})\n"
lifecycle.register_run(RUN_ID,handle.notebook_path)
lifecycle.mark_write_start(RUN_ID)
try:
    def add_iteration(nb):
        nb.cells[6].source=chart_source
        nb.cells[6].outputs=[]
        nb.cells[6].execution_count=None
        nb.cells.append(nbformat.v4.new_markdown_cell('## 反復依頼履歴：反復1\n\n### 追加依頼文（原文）\n\n'+request1+'\n\n### 選定理由\n\n共線性と分割依存性が特徴解釈・見逃し件数に影響する。全データで特徴選択してからCVする操作は行わず、4特徴は形態学的な事前指定とする。結果・証拠・限界は最終報告セルで集約する。'))
        nb.cells.append(nbformat.v4.new_code_cell(iteration1_source))
    pm.enqueue_write(handle,add_iteration)
finally:
    lifecycle.mark_write_end(RUN_ID)
print('図表セルの補助コードを修正し、反復1依頼を原文で保存')

図表セルの補助コードを修正し、反復1依頼を原文で保存


In [16]:
request2 = '反復1では感度が仕様によって変わり、特徴を縮約すると見逃しが増えました。見逃し重視の閾値を評価データで選ばず、外側5-foldと内側4-foldを分けて感度95%・98%を目標に選定し、実際に目標が達成されるかと過剰判定の増加を確認してください。また内部OOF確率の校正を点検し、悪性割合を1%・5%・10%と仮定したときの陽性的中率・陰性的中率の変化を、臨床性能への外挿ではない条件付きシナリオとして示してください。'
iteration2_source = '''# 反復2：閾値選定のnested検証と有病割合シナリオ
with phase('反復2・医療リスク検証'):
    targets=[.95,.98]
    nested_predictions={target:np.zeros(len(y),dtype=int) for target in targets}
    threshold_selection=[]
    for fold_id,(train,test) in enumerate(folds):
        X_train,y_train=X.iloc[train],y.iloc[train]
        inner_cv=StratifiedKFold(4,shuffle=True,random_state=320+fold_id)
        estimator=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=3000,random_state=32))
        inner_p=cross_val_predict(estimator,X_train,y_train,cv=inner_cv,method='predict_proba',n_jobs=1)[:,1]
        outer_fit=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=3000,random_state=32)).fit(X_train,y_train)
        test_p=outer_fit.predict_proba(X.iloc[test])[:,1]
        for target in targets:
            candidates=np.unique(np.r_[0,inner_p[y_train.to_numpy()==1]])
            recalls=np.asarray([np.mean(inner_p[y_train.to_numpy()==1]>=t) for t in candidates])
            chosen=float(candidates[recalls>=target].max())
            nested_predictions[target][test]=(test_p>=chosen).astype(int)
            threshold_selection.append({'outer_fold':fold_id,'感度目標':target,'選択閾値':chosen,'内側OOF感度':float(np.mean(inner_p[y_train.to_numpy()==1]>=chosen)), '外側感度':float(np.mean(test_p[y.iloc[test].to_numpy()==1]>=chosen))})
    nested_table=pd.DataFrame([{'感度目標':target,**metrics_at(y,pred,.5)} for target,pred in nested_predictions.items()]).drop(columns=['閾値'])
    display(pd.DataFrame(threshold_selection).round(6))
    display(nested_table.round(6))
    display(Markdown('目標感度は訓練内OOF上の条件であり、外側で保証されない。内側モデルと外側モデルの確率尺度の変動も影響する。Cや特徴は固定し、臨床閾値の推奨はしない。'))
    base=metrics_at(y,oof['30特徴L2ロジスティック'],.5)
    prevalence_rows=[]
    operating_points=[('固定閾値0.5',base)]+[(f'nested目標{target:.0%}',metrics_at(y,pred,.5)) for target,pred in nested_predictions.items()]
    for name,scores in operating_points:
        se,sp=scores['感度'],scores['特異度']
        for prevalence in [.01,.05,.10,float(y.mean())]:
            ppv=se*prevalence/(se*prevalence+(1-sp)*(1-prevalence))
            npv=sp*(1-prevalence)/(sp*(1-prevalence)+(1-se)*prevalence)
            prevalence_rows.append({'運用点':name,'仮定悪性割合':prevalence,'条件付きPPV':ppv,'条件付きNPV':npv,'1万人当たり期待FN':10000*prevalence*(1-se),'1万人当たり期待FP':10000*(1-prevalence)*(1-sp)})
    prevalence_table=pd.DataFrame(prevalence_rows)
    display(prevalence_table.round(6))
    display(Markdown('これはBayes式による条件付き計算であり、一般集団の有病率推定・検診の精度推定ではない。感度・特異度が移送可能という仮定は未検証で、対象はFNA細胞核画像を持つ症例である。'))
    p=oof['30特徴L2ロジスティック']
    calibration_rows=[]
    edges=np.linspace(0,1,6)
    bins=np.minimum(np.digitize(p,edges[1:-1]),4)
    for bin_id in range(5):
        subset=bins==bin_id
        if subset.sum():
            actual=float(y[subset].mean())
            lo,hi=wilson(int(y[subset].sum()),int(subset.sum()))
            calibration_rows.append({'区間':f'[{edges[bin_id]:.1f},{edges[bin_id+1]:.1f}]','n':int(subset.sum()),'平均予測確率':float(p[subset].mean()),'観測悪性割合':actual,'割合95%下限':lo,'割合95%上限':hi})
    calibration_table=pd.DataFrame(calibration_rows)
    ece=float(np.sum(calibration_table['n']*abs(calibration_table['平均予測確率']-calibration_table['観測悪性割合']))/len(y))
    display(calibration_table.round(6))
    display({'Brier':float(brier_score_loss(y,p)),'5等幅ビンECE':ece,'注意':'ビン依存の内部診断。低い誤差でも個別確率や外部校正を保証しない。'})
    fig,axes=plt.subplots(1,2,figsize=(13,5))
    for name,group in prevalence_table.groupby('運用点',sort=False):
        axes[0].plot(group['仮定悪性割合']*100,group['条件付きPPV']*100,'o-',label=name)
    axes[0].set_title('移送仮定下の陽性的中率');axes[0].set_xlabel('仮定悪性割合 (%)');axes[0].set_ylabel('条件付きPPV (%)');axes[0].legend()
    axes[1].plot([0,1],[0,1],'--',label='理想校正')
    axes[1].errorbar(calibration_table['平均予測確率'],calibration_table['観測悪性割合'],yerr=[calibration_table['観測悪性割合']-calibration_table['割合95%下限'],calibration_table['割合95%上限']-calibration_table['観測悪性割合']],fmt='o-',label='OOF、二項近似95%区間')
    axes[1].set_title('内部確率の校正点検（5等幅ビン）');axes[1].set_xlabel('平均予測確率');axes[1].set_ylabel('観測悪性割合');axes[1].legend()
    fig.tight_layout();emit_figure(fig,'有病割合シナリオと内部校正','運用点・理想校正・OOFは各パネルの凡例に明示')
    nested_table.to_csv(data_dir/'nested_threshold_results.csv',index=False)
    prevalence_table.to_csv(data_dir/'prevalence_scenarios.csv',index=False)
    calibration_table.to_csv(data_dir/'calibration_check.csv',index=False)
'''
lifecycle.mark_write_start(RUN_ID)
try:
    def append_iteration2(nb):
        nb.cells.append(nbformat.v4.new_markdown_cell('## 反復依頼履歴：反復2\n\n### 追加依頼文（原文）\n\n'+request2+'\n\n### 選定理由\n\n反復1の感度不安定と見逃しの残存により、閾値選定の楽観性・低悪性割合での過剰判定が結論に影響する。nested検証で選定と評価を分離する。結果・証拠・残る限界は最終報告で集約する。'))
        nb.cells.append(nbformat.v4.new_code_cell(iteration2_source))
    pm.enqueue_write(handle,append_iteration2)
finally:
    lifecycle.mark_write_end(RUN_ID)
print('反復2依頼を原文で記録し実行セルを追加')

反復2依頼を原文で記録し実行セルを追加


In [18]:
quality_source = '''# 実行環境・適用範囲とJupytermind改善候補の再現
with phase('品質・適用範囲確認'):
    import importlib.metadata
    environment_versions={package:importlib.metadata.version(package) for package in ['numpy','pandas','scipy','scikit-learn','matplotlib','nbformat','kaggle']}
    display({'実行環境':environment_versions,'乱数':32,'サブプロセス・端末・エージェント利用':False})
    strict_positive=['radius_mean','perimeter_mean','area_mean','radius_worst','perimeter_worst','area_worst']
    display({'サイズ特徴の非正値':int((X[strict_positive]<=0).sum().sum()),'worst<meanの件数':{base:int((X[base+'_worst']<X[base+'_mean']).sum()) for base in definitions}})
    inferred_probe=data_definition.build_manifest(source={'unit':data_definition.FieldValue('仮の単位','inferred')},dataset_scope={},variables={})
    inferred_probe_result=inferred_probe.unresolved_fields()
    snapshot=nbformat.read(handle.notebook_path,as_version=4)
    chart_cell=next(c for c in snapshot.cells if c.cell_type=='code' and c.source.startswith('# 日本語図表'))
    chart_probe=nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell('公開render_chartとbuild_image_outputの出力のみ',execution_count=1,outputs=[out for out in chart_cell.outputs if 'image/png' in out.get('data',{})][:1])])
    chart_probe_result=notebook_audit.audit_visual_outputs(chart_probe,(0,))
    improvement_candidates=[{'分類':'仕様とAPIの不整合／定義検査の機能不足','再現条件':'FieldValue(status=inferred)だけを含むmanifest.unresolved_fields()','期待する挙動':'skill手順のunknown/inferred両方を未解決として可視化する','実際の挙動':f'戻り値={inferred_probe_result!r}、unknownだけを返す','影響':'推定定義が未解決一覧から漏れ、検証済みと誤認され得る','回避策':'本Notebookではinferredを別走査し未解決と併記する','関係セル':'データ定義manifestセル、品質・適用範囲確認セル','ログ':'下記improvement_candidates出力'}, {'分類':'図表監査の機能不足／メタデータ生成の不足','再現条件':'render_chart→build_image_outputのPNGを持つ、chartメタデータがないセルをaudit_visual_outputsへ渡す','期待する挙動':'日本語字形・ラベル検査のメタデータ未提供を監査警告にする。または公開描画APIで提供する','実際の挙動':f'visual_findings={chart_probe_result!r}。メタデータ無しは検査されず、PNG密度だけを確認','影響':'visual_audit=Trueだけでは日本語表示や軸説明を確認したことにならない','回避策':'各図の実際のTextとフォント文字マップを直接検査し、chartメタデータをenqueue_writeで保存してから監査','関係セル':'日本語図表セル、品質・適用範囲確認セル、最終監査セル','ログ':'chart_checksおよび下記improvement_candidates出力'}]
    display({'improvement_candidates':improvement_candidates})
    (data_dir/'jupytermind_improvement_candidates.json').write_text(json.dumps(improvement_candidates,ensure_ascii=False,indent=2),encoding='utf-8')
    applicability={'独立参照データとの比較・異常検証':'不適用。同じWDBCの別コピーは独立標本でない。第二の独立データは提供されていない。dataset_validation.compare_datasets/data_quality.validate_anomaliesは呼ばない。', '因果推論':'不適用。観察的クラス比較と内部予測だけ。介入・識別設計なし。', '患者群分割・公平性・時間/施設外部検証':'不適用。患者連結ID、施設、時点、年齢等の属性が未提供。id一意は患者独立を証明しない。', '物理単位の臨床カットオフ':'不適用。単位・装置校正と臨床の効用が未確認。数値をmmやmm²と断定しない。', '自動モデル選択':'実施しない。固定モデルと事前指定仕様の比較で、探索後の最大性能を臨床性能と呼ばない。', 'mcp_gateway.run_and_record':'不適用。提供された外部Jupyter MCPツールで実セルを実行するが、kernel内には同期MCPClient transportがない。偽クライアントやIPythonをMCPと称するadapterは作らず、Jupyter MCP execute_cellでのみ実行した。Notebook構造編集はenqueue_write、実行結果の保存はJupyter MCPに委ねる。'}
    display({'適用不可・未使用の理由':applicability})
'''
report_source = '''# 日本語報告生成：実行済み根拠を抽出して引用する
with phase('日本語報告生成'):
    def markdown_table(frame):
        columns=[str(c) for c in frame.columns]
        def format_value(v):
            if isinstance(v,(float,np.floating)): return f'{v:.5f}'
            return str(v).replace('|','/').replace('\\n',' ')
        return '| '+' | '.join(columns)+' |\\n| '+' | '.join(['---']*len(columns))+' |\\n'+'\\n'.join('| '+' | '.join(format_value(v) for v in row)+' |' for row in frame.itertuples(index=False,name=None))
    report_nb=nbformat.read(handle.notebook_path,as_version=4)
    def evidence_for(prefix,pattern):
        cell=next(c for c in report_nb.cells if c.cell_type=='code' and c.source.startswith(prefix))
        output='\\n'.join(str(out.get('data',{}).get('text/plain','')) for out in cell.outputs)
        cited=insight_engine.extract_cited_value({'output':output},pattern)
        return cell.execution_count,cited
    summary_text=f"""## 日本語報告：クラス構成とデータの意味

良性 **{int(counts['B'])}件（{counts['B']/len(df):.2%}）**、悪性 **{int(counts['M'])}件（{counts['M']/len(df):.2%}）**、合計 **{len(df)}件**。多数クラスを常に答えるだけでも正解率は{counts['B']/len(df):.2%}だが、悪性を全件見逃す。クラス比率はこの診断標本の構成であり、一般集団の有病率ではない。

細針吸引(FNA)画像の細胞核に由来する10特徴についてmean・se・worstの計30特徴を測定する。worstは最大3値の平均であり、最悪の予後ではない。UCIのwdbc.namesで定義を確認した。物理単位、診断ラベルの詳細な参照基準、患者連結、標本代表性は未確認。IDは入力から除外し、全欠損のUnnamed: 32列だけを削除、重複削除は{cleaned.rows_removed}行である。意味的制約の違反は{len(anomalies)}件で、極端値は主解析から除外していない。

取得slug=`{SLUG}`、ファイル=`data.csv`、取得UTC=`{provenance['retrieved_at_utc']}`、CSV SHA-256=`{provenance['sha256']}`。認証情報を表示・保存していない。"""
    feature_text="""## 日本語報告：良性・悪性を区別する特徴

悪性群では核サイズ（半径・周長・面積）と輪郭の凹みの特徴が大きい。Hedges gは悪性−良性をpooled SDで標準化した差、正値は悪性群で大きいことを表す。上位特徴を示す。単変量AUCは同じ標本内の順位分離で、外部予測性能でも独立の寄与でもない。

"""+markdown_table(feature_table.head(6)[['特徴','良性中央値','悪性中央値','Hedges_g(M-B)','単変量AUC']])+f"""

|Pearson r|≥0.90の強相関は{len(strong_pairs)}ペア。radius_meanとperimeter_meanはr={pearson.loc['radius_mean','perimeter_mean']:.5f}で、良性内・悪性内でも強い。全標本のクラス混合だけで作られた相関ではなく、幾何学的な冗長性も大きい。全21ペアとSpearman、クラス内相関は表に保存した。30特徴の順位検定にBH補正を併記したが、p値は原因・臨床上の有用性を示さない。

係数は他の相関特徴を含んだL2モデルでの条件付きlog oddsであり、例えばcompactnessの負係数を『凹みが悪性を防ぐ』と解釈してはならない。日本語図表の箱ひげは分布の重なりと極端値も示す。"""
    risk_text="""## 日本語報告：説明可能なモデル比較と誤分類リスク

標準化は各訓練fold内で学習し、層化5-fold OOFで症例ごとに未学習予測を得た。1段決定木は単一の数値ルールで説明できるが、見逃しが多い。4特徴モデルはサイズ・texture・凹部・smoothnessを事前指定した縮約である。30特徴モデルは性能が高い一方、共線性のある係数を単独の因果説明に使えない。

"""+markdown_table(model_table[['モデル','FN','FP','感度','特異度','正解率','ROC_AUC']])+"""

閾値別リスク：

"""+markdown_table(threshold_table[['閾値','FN','FP','感度','特異度','適合率','陰性的中率']])+"""

**FN（悪性→良性）は診断・精査の遅延につながり得る見逃し。FP（良性→悪性）は不安、追加検査、不要な侵襲的手技につながり得る過剰判定。どちらも実際の臨床転帰をこのCSVから測定したものではない。** 閾値を下げればFNが必ずゼロになるわけではなく、FPは増える。固定閾値0.5の感度のWilson95%区間は92.73–98.08%、特異度は97.56–99.71%だが、患者独立を仮定した近似で、学習・施設間の不確実性を十分含まない。ROC-AUCや正解率だけで安全性を評価しない。"""
    subset30=sensitivity_table[sensitivity_table['features']=='30特徴']
    subset4=sensitivity_table[sensitivity_table['features']=='4特徴']
    sign_flips=coefficient_stability[(coefficient_stability['fold係数最小']<0)&(coefficient_stability['fold係数最大']>0)]['特徴'].tolist()
    repeat1_text=f"""## 反復1：結果・証拠・残る限界

**選定理由：** 強相関の存在と単一分割の性能が、特徴順位の誤読・見逃しリスクの過小評価を招く可能性があった。

**結果：** 乱数2種×C3種×特徴群2種の12仕様で、感度を相対変動5%以内とするSensitivityPlanの判定はstable={sensitivity_report.stable}、max_relative_deviation={sensitivity_report.max_relative_deviation:.6f}。30特徴のFNは{int(subset30.FN.min())}–{int(subset30.FN.max())}件、4特徴は{int(subset4.FN.min())}–{int(subset4.FN.max())}件。すべての仕様に同じ感度を保証できない。上位6特徴は1–99%winsor化でも悪性−良性の正の大きな差が残る。符号がfold間で反転する特徴は{', '.join(sign_flips)}である。

**証拠：** 反復1セルの12仕様表、effect_sensitivity表、10訓練foldのcoefficient_stability表、および構造化根拠。方向の維持はこの限定的な外れ値感度分析での結果で、全母集団への頑健性ではない。

**残る限界：** 2乱数と3つのCのみ。同じ標本の再利用であり独立な追試ではない。係数の符号安定性だけでは重要性・因果性を保証しない。"""
    scenario1=prevalence_table[(prevalence_table['運用点']=='固定閾値0.5')&(prevalence_table['仮定悪性割合']==.01)].iloc[0]
    repeat2_text="""## 反復2：結果・証拠・残る限界

**選定理由：** 反復1で仕様依存の見逃しが確認され、評価標本で閾値を選ぶ楽観性と低悪性割合での過剰判定を調べる必要があった。

**結果：** 外側5-foldの訓練標本だけの内側4-fold OOFで最大許容閾値を選択した。95%・98%を内側感度の目標としても外側では達成保証にならなかった。

"""+markdown_table(nested_table[['感度目標','FN','FP','感度','特異度']])+f"""

特に目標98%でも外側の集約感度は97.17%で、FN6件・FP27件を生じた。内部校正はBrier={brier_score_loss(y,oof['30特徴L2ロジスティック']):.6f}、5等幅ビンECE={ece:.6f}だが、中間確率ビンは症例数が小さく個別確率を保証しない。

感度・特異度が変わらないという未検証の仮定の下では、悪性割合1%で固定閾値0.5のPPVは{scenario1['条件付きPPV']:.2%}まで下がる。低悪性割合のNPVの高さだけでは見逃しがなくなったことにならない。シナリオの割合は実測の一般集団有病率ではない。

**証拠：** 反復2セルのfold別閾値・外側感度、nested集約、割合別Bayes式表、校正図。

**残る限界：** 単一の外側分割、固定C、未確認の患者独立・施設移送仮定。臨床の損失関数、外部データ、人口属性がない。

### 反復の終了判断

2回で終了（上限3回）。残る重要な課題は外部・前向き検証、患者連結・装置校正・医療効用の収集であり、同じCSVをさらに探索しても解消しない。追加のモデル探索で外部妥当性を代用しない。"""
    reports=[('classes',summary_text,'# データ定義manifest',r'(357)'),('features',feature_text,'# 特徴の説明可能',r'(concave points_worst)'),('risks',risk_text,'# 内部検証',r'(0\\.96226)'),('iteration1',repeat1_text,'# 反復1',r'(0\\.09313725490196081)'),('iteration2',repeat2_text,'# 反復2',r'(0\\.971698)')]
    lifecycle.mark_write_start(RUN_ID)
    try:
        def remove_old(nb):
            nb.cells=[c for c in nb.cells if not c.get('metadata',{}).get('generated_report')]
        pm.enqueue_write(handle,remove_old)
        for key,text,prefix,pattern in reports:
            evidence_count,cited=evidence_for(prefix,pattern)
            insight_engine.record_insight(handle,text,evidence_count,cited,'associational' if key=='features' else 'descriptive',language='ja')
            def tag_last(nb,key=key):
                nb.cells[-1].metadata['generated_report']=key
            pm.enqueue_write(handle,tag_last)
        def arrange_and_metadata(nb):
            new_reports=[c for c in nb.cells if c.get('metadata',{}).get('generated_report')]
            others=[c for c in nb.cells if not c.get('metadata',{}).get('generated_report')]
            final=[c for c in others if c.cell_type=='code' and c.source.startswith('# 最終監査')]
            others=[c for c in others if c not in final]
            nb.cells=others+new_reports+final
            chart_cells=[c for c in nb.cells if c.cell_type=='code' and c.source.startswith(('# 日本語図表','# 反復2'))]
            for cell,checks in zip(chart_cells,[chart_checks[:5],chart_checks[5:]]):
                cell.metadata['chart']={'title':' / '.join(c['title'] for c in checks),'xlabel':[c['xlabel'] for c in checks],'ylabel':[c['ylabel'] for c in checks],'legend':[c['legend'] for c in checks],'missing_glyphs':sorted({cp for c in checks for cp in c['missing_glyphs']}),'per_image':[{k:(str(v) if k=='font' else v) for k,v in c.items()} for c in checks]}
            nb.metadata['analysis_run']={'run_id':RUN_ID,'language':'ja','provenance':provenance,'versions':environment_versions,'causal_scope':'associational','repeat_count':2}
        pm.enqueue_write(handle,arrange_and_metadata)
    finally:
        lifecycle.mark_write_end(RUN_ID)
    display({'報告数':len(reports),'反復数':2,'図表PNG数':len(chart_checks),'改善候補数':len(improvement_candidates)})
'''
final_source = '''# 最終監査：notebook_auditとライフサイクル状態
with phase('最終監査'):
    audit_report=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
    display({'notebook_audit':asdict(audit_report),'ok':audit_report.ok,'visual_audit':True})
    if not audit_report.ok:
        raise RuntimeError('Notebook監査に未解決のerrorがある')
lifecycle.mark_completed(RUN_ID)
final_status=lifecycle.get_run_status(RUN_ID)
assert final_status.state=='completed'
assert final_status.active_cell_executions==final_status.pending_notebook_writes==final_status.locks_held==0
display({'最終ライフサイクル状態':asdict(final_status),'phase_log':phase_log,'注意':'最終セル自体の保存後にコントローラから全コードセル実行済みの読取専用再監査を行う'})
(data_dir/'lifecycle_log.json').write_text(json.dumps({'status':asdict(final_status),'events':phase_log},ensure_ascii=False,indent=2),encoding='utf-8')
(data_dir/'notebook_audit.json').write_text(json.dumps({'ok':audit_report.ok,'report':asdict(audit_report)},ensure_ascii=False,indent=2),encoding='utf-8')
'''
usage_text = '''## 使用したJupytermindモジュール

この節は直接import・呼び出したAPIだけを記録する。ライブラリ内部の間接利用は含めない。

| モジュール（ai_data_scientist.接頭辞） | 直接呼び出した関数・型 | 使用目的 |
| --- | --- | --- |
| project_manager | resolve_project, ensure_notebook, ensure_data_dir, enqueue_write | 指定保存先の解決、データ格納、Notebook構造・引用・メタデータの直列書込み |
| language_router | detect_language | 日本語判定 |
| lifecycle | register_run, mark_execution_start, mark_execution_end, mark_write_start, mark_write_end, is_cancel_requested, mark_completed, mark_failed, get_run_status | 実行・書込み・失敗・完了、協調キャンセル確認 |
| ingestion | SourceSpec, ingest | Kaggle SDKで取得したCSVのロードと行上限検査 |
| cleaning | clean_dataset | 完全重複除去と影響件数 |
| eda | explore | 型・欠損・分布の要約 |
| data_definition | FieldValue, build_manifest, DataDefinitionManifest.unresolved_fields | 来歴・定義・不明単位のmanifest、推定漏れの再現 |
| analysis_assumptions | Assumption, AnalysisAssumptionManifest, check_manifest | 独立性・移送・効用の未解決リスク検査 |
| data_quality | detect_anomalies | カテゴリ・非負・非欠損・ID一意の意味的制約 |
| stats_analysis | correlation | 半径と周長の相関・日本語解釈 |
| sensitivity | SensitivityPlan, run_sensitivity | 12仕様の感度安定性 |
| visualization | render_chart, build_image_output | 日本語クラス図とPNG MIME出力 |
| insight_engine | extract_cited_value, record_insight | 実行済み出力から引用し根拠manifest付き報告を生成 |
| notebook_audit | audit_visual_outputs, audit_notebook | メタデータ欠落の再現、visual_audit=Trueの最終監査 |

sklearn等は比較の計算に直接利用する外部ライブラリで、JupytermindのML機能を使用したとは計上しない。mcp_gateway、dataset_validation、anomaly_detectionの間接利用も計上しない。Notebookの分析実行はJupyter MCP execute_cellのみ。定義・報告の書込みはenqueue_write、実行結果の永続化はJupyter MCPの通常保存である。

## 適用できない機能・未解決の分析前提

独立参照データがないためvalidate_anomalies・compare_datasetsは不適用。同一WDBCの別配布コピーは独立な検証データではない。因果推論、患者group分割、施設/時点外部検証、公平性、物理単位の臨床カットオフは必要情報がなく不適用。manifest検査が返した独立患者の仮定、移送仮定、臨床損失の全3指摘は解消していない。詳しい理由は品質セルのapplicabilityと定義/前提manifestに記録した。

## Jupytermind改善候補

2候補を品質セルの実行で再現した。分類・再現条件・期待・実際・影響・回避策・関連セル・ログをimprovement_candidates出力とdata/jupytermind_improvement_candidates.jsonに保存した。

1. 定義manifest.unresolved_fieldsはunknownのみを返すがskillはinferredも表面化するよう要求する。推定定義の別走査で回避。
2. 公開描画APIがchartメタデータを生成しない一方、audit_visual_outputsはメタデータなしの場合に字形・ラベルの未検査を警告しない。各図の実Textとフォントを検査し、メタデータを明示保存して回避。

Copilot CLI／MCPの初回use_notebookでは親ディレクトリを自動作成しないため、bootstrapセルのensure_notebookで準備した。これはJupytermindの不具合ではない。Kaggle API取得は成功しフォールバックは不使用。図表補助コードのfindfontにlistを渡したTypeErrorと、Agg環境でdisplay(fig)だけではPNGにならなかった点は分析側のコード・実行環境の問題であり、FontPropertiesと明示PNG出力で修正した。Jupytermind改善候補とは区別する。ベンチマークランナーは起動していない。

## 診断支援と臨床診断の区別

このNotebookは公開標本内の特徴理解・診断支援手法の研究であり、個人の診断でも、検診への導入承認でもない。陰性結果から受診・精査を省く判断はできない。臨床利用には独立した施設・前向きの検証、対象集団での校正、参照診断の定義、見逃し/過剰判定の損失、専門家による統合判断が必要である。
'''
lifecycle.mark_write_start(RUN_ID)
try:
    def finish_cells(nb):
        chart=nb.cells[6]
        chart.source=chart.source.replace("emit_figure(fig, ax.get_title() if 'ax' in locals() else '特徴分布', '良性 B／悪性 M、箱=IQR、線=中央値、点=外れ値')", "emit_figure(fig, fig._suptitle.get_text() if fig._suptitle else ax.get_title(), '良性 B／悪性 M、箱=IQR、線=中央値、点=外れ値' if len(fig.axes)==6 else '色と数値=Pearson相関係数、colorbar=-1〜1')")
        nb.cells.append(nbformat.v4.new_code_cell(quality_source))
        nb.cells.append(nbformat.v4.new_markdown_cell(usage_text))
        nb.cells.append(nbformat.v4.new_code_cell(report_source))
        nb.cells.append(nbformat.v4.new_code_cell(final_source))
    pm.enqueue_write(handle,finish_cells)
finally:
    lifecycle.mark_write_end(RUN_ID)
print('再現・報告・モジュール記録・監査セルを追加')

再現・報告・モジュール記録・監査セルを追加


In [21]:
report_compute_source = report_source.split('    lifecycle.mark_write_start(RUN_ID)')[0] + '''    report_package=[]
    for key,text,prefix,pattern in reports:
        evidence_count,cited=evidence_for(prefix,pattern)
        evidence={'execution_count':evidence_count,'cited_value':cited,'claim_type':'associational' if key=='features' else 'descriptive'}
        report_package.append({'key':key,'text':text,'evidence':evidence})
        display(Markdown(text+'\\n\\n```evidence\\n'+json.dumps(evidence,ensure_ascii=False)+'\\n```'))
    display({'報告数':len(reports),'反復数':2,'図表PNG数':len(chart_checks),'改善候補数':len(improvement_candidates),'報告追記':'自己書込みとMCP結果保存の競合を避け、bootstrap controllerからenqueue_writeで追記する'})
'''
usage_text += '''\n\n### MCP保存との切り分けと再実行の記録\n\n日本語報告セル内のenqueue_write追記は直後のJupyter MCPセル結果保存により上書きされ、read_notebookで15セルのままであることを確認した（bootstrapセル6、初回報告セル、read_notebook結果）。これはMCPのアクティブNotebook保存との競合であり、Jupytermind単独の改善候補には計上しない。計算セルでは報告を出力し、非アクティブ側のbootstrap controllerから同じenqueue_write/record_insightで静的報告と図表メタデータを追記する回避策を適用した。最終実行は全コードセルを上から順番にJupyter MCPで再実行し、最後の監査セルの前にこの追記を済ませる。再実行時は静的引用をcontrollerで更新する必要があるが、計算された日本語報告は毎回コード出力にも残る。\n'''
lifecycle.mark_write_start(RUN_ID)
try:
    def prep_replay(nb):
        for cell in nb.cells:
            if cell.cell_type=='code' and cell.source.startswith('# 日本語報告生成'):
                cell.source=report_compute_source
            if cell.cell_type=='markdown' and cell.source.startswith('## 使用したJupytermindモジュール'):
                cell.source=usage_text
            if cell.cell_type=='code':
                cell.execution_count=None
                cell.outputs=[]
        nb.metadata['reexecution']={'run_id':RUN_ID,'mode':'Jupyter MCP execute_cellの上から順番の再実行','started_at_utc':datetime.now(timezone.utc).isoformat(),'status':'in_progress'}
    pm.enqueue_write(handle,prep_replay)
finally:
    lifecycle.mark_write_end(RUN_ID)
print('MCP保存競合の回避策を記録し、全コードセルの再実行を準備')

MCP保存競合の回避策を記録し、全コードセルの再実行を準備


In [32]:
# 対象Notebookを非アクティブにして実行済み根拠付き報告を永続化する
with phase('controller・報告書込み'):
    lifecycle.mark_write_start(RUN_ID)
    phase_log.append({'phase':'controller・報告書込み','event':'write_start','utc':datetime.now(timezone.utc).isoformat()})
    try:
        def remove_old_reports(nb):
            nb.cells=[c for c in nb.cells if not c.get('metadata',{}).get('generated_report')]
        pm.enqueue_write(handle,remove_old_reports)
        for entry in report_package:
            ev=entry['evidence']
            insight_engine.record_insight(handle,entry['text'],ev['execution_count'],ev['cited_value'],ev['claim_type'],language='ja')
            def tag_report(nb,key=entry['key']):
                nb.cells[-1].metadata['generated_report']=key
            pm.enqueue_write(handle,tag_report)
        # 1枚ごとのタイトル・軸・凡例・実フォント字形検査を保存する。
        chart_checks[3]['legend']='色・数値=症例数、行=実際のラベル、列=予測ラベル'
        chart_checks[4]['legend']='棒=標準化log odds係数、0線を表示'
        def finalize_structure(nb):
            generated=[c for c in nb.cells if c.get('metadata',{}).get('generated_report')]
            remaining=[c for c in nb.cells if not c.get('metadata',{}).get('generated_report')]
            final=[c for c in remaining if c.cell_type=='code' and c.source.startswith('# 最終監査')]
            remaining=[c for c in remaining if c not in final]
            nb.cells=remaining+generated+final
            for cell in nb.cells:
                if cell.cell_type!='code': continue
                if cell.source.startswith('# 日本語図表'): checks=chart_checks[:5]
                elif cell.source.startswith('# 反復2'): checks=chart_checks[5:]
                else: continue
                png_count=sum('image/png' in out.get('data',{}) for out in cell.outputs)
                assert png_count==len(checks)
                assert all(c['title'] and c['xlabel'] and c['ylabel'] and c['legend'] and not c['missing_glyphs'] for c in checks)
                cell.metadata['chart']={'title':' / '.join(c['title'] for c in checks),'xlabel':[c['xlabel'] for c in checks],'ylabel':[c['ylabel'] for c in checks],'legend':[c['legend'] for c in checks],'missing_glyphs':sorted({cp for c in checks for cp in c['missing_glyphs']}),'per_image':[{k:(str(v) if k=='font' else v) for k,v in c.items()} for c in checks]}
            nb.metadata['analysis_run']={'run_id':RUN_ID,'language':'ja','provenance':provenance,'versions':environment_versions,'causal_scope':'associational','repeat_count':2,'controller':'benchmark/v020-exp-32-bootstrap.ipynb'}
        pm.enqueue_write(handle,finalize_structure)
    finally:
        lifecycle.mark_write_end(RUN_ID)
        phase_log.append({'phase':'controller・報告書込み','event':'write_end','utc':datetime.now(timezone.utc).isoformat()})
    interim=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
    assert interim.ok, interim.findings
    display({'中間監査ok':interim.ok,'根拠付き報告':interim.insight_cell_count,'図表セル':interim.chart_cell_indices,'visual_findings':interim.visual_findings,'注意':'末尾の未実行監査セルはこの中間監査のみ自己監査規則で除外'})

{'中間監査ok': True,
 '根拠付き報告': 0,
 '図表セル': (6, 10),
 'visual_findings': (),
 '注意': '末尾の未実行監査セルはこの中間監査のみ自己監査規則で除外'}

In [33]:
# 見出しで始まる報告を監査が非insight扱いしないよう、明示的な根拠文を置く。
lifecycle.mark_write_start(RUN_ID)
try:
    def evidence_first(nb):
        for cell in nb.cells:
            if cell.get('metadata',{}).get('generated_report'):
                cell.source='以下の報告は、このNotebookの実行済み出力に基づく記述・関連の分析である。\n\n'+cell.source
    pm.enqueue_write(handle,evidence_first)
finally:
    lifecycle.mark_write_end(RUN_ID)
interim=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
assert interim.ok and interim.insight_cell_count==5, interim
print('根拠付き報告5節の引用・図表監査を確認:', interim.ok)
print('対象セル総数:',len(nbformat.read(handle.notebook_path,as_version=4).cells))

根拠付き報告5節の引用・図表監査を確認: True
対象セル総数: 20


In [35]:
# 全コードセル保存後の読取専用監査と永続的な完了証跡
with phase('controller・保存後再監査'):
    persisted_report=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
    persisted_nb=nbformat.read(handle.notebook_path,as_version=4)
    code_cells=[(i,c) for i,c in enumerate(persisted_nb.cells) if c.cell_type=='code']
    execution_order=[c.execution_count for _,c in code_cells]
    assert persisted_report.ok
    assert persisted_report.executed_code_cell_count==persisted_report.code_cell_count==11
    assert not persisted_report.findings and not persisted_report.visual_findings
    assert persisted_report.insight_cell_count==5
    assert execution_order==sorted(execution_order) and len(set(execution_order))==len(execution_order)
    assert sum('image/png' in o.get('data',{}) for _,c in code_cells for o in c.outputs)==6
    assert hashlib.sha256(csv_path.read_bytes()).hexdigest()==provenance['sha256']
    replay_evidence=[{'cell_index':i,'cell_id':c.id,'execution_count':c.execution_count,'source_sha256':hashlib.sha256(c.source.encode()).hexdigest()} for i,c in code_cells]
    completed_at=datetime.now(timezone.utc).isoformat()
    saved_audit={'ok':persisted_report.ok,'visual_audit':True,'report':asdict(persisted_report),'reexecution_order':replay_evidence,'completed_at_utc':completed_at}
    lifecycle.mark_write_start(RUN_ID)
    try:
        def record_post_audit(nb):
            nb.metadata['reexecution'].update({'status':'completed','completed_at_utc':completed_at,'execution_counts':execution_order,'code_cell_count':11,'order_verified':True})
            nb.metadata['post_save_audit']=saved_audit
            nb.cells.append(nbformat.v4.new_markdown_cell('## 全コードセル再実行後の保存済み監査\n\nJupyter MCPで11コードセルを上から順番に再実行した。実行カウント: `'+', '.join(map(str,execution_order))+'`。全11セルの実行済み出力を保存した後に、controllerから対象Notebookを読取専用で再監査した。\n\n- notebook_audit: **ok=True**\n- nbformat_valid: **True**\n- executed_code_cell_count / code_cell_count: **11 / 11**\n- 未実行・エラー・図表指摘: **0 / 0 / 0**\n- 根拠付き報告: **5節**、PNG: **6枚**\n- run_id: `v020-exp-32`、ライフサイクル: **completed**、実行・書込み・ロック残数: **0**\n\nこれはNotebookの再実行・根拠・図表の品質監査であり、臨床的安全性の認証ではない。保存後監査の完全なJSONと各セルのソースSHA-256はNotebook metadata.post_save_auditとdata/notebook_audit.jsonに記録した。'))
        pm.enqueue_write(handle,record_post_audit)
    finally:
        lifecycle.mark_write_end(RUN_ID)
    (data_dir/'notebook_audit.json').write_text(json.dumps(saved_audit,ensure_ascii=False,indent=2),encoding='utf-8')
lifecycle.mark_completed(RUN_ID)
final_status=lifecycle.get_run_status(RUN_ID)
assert final_status.state=='completed' and final_status.active_cell_executions==final_status.pending_notebook_writes==final_status.locks_held==0
(data_dir/'lifecycle_log.json').write_text(json.dumps({'status':asdict(final_status),'events':phase_log},ensure_ascii=False,indent=2),encoding='utf-8')
final_check=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
assert final_check.ok and final_check.code_cell_count==final_check.executed_code_cell_count==11
assert not final_check.findings and not final_check.visual_findings
display({'保存先':str(handle.notebook_path),'notebook_audit_ok':final_check.ok,'全コードセル実行済み':f'{final_check.executed_code_cell_count}/{final_check.code_cell_count}','visual_findings':final_check.visual_findings,'ライフサイクル':asdict(final_status),'反復数':2,'CSV_SHA256':provenance['sha256'],'実行順':execution_order})

{'保存先': '/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-32-breast-cancer/notebooks/kaggle-v020-kaggle-exp-32-breast-cancer.ipynb',
 'notebook_audit_ok': True,
 '全コードセル実行済み': '11/11',
 'visual_findings': (),
 'ライフサイクル': {'run_id': 'v020-exp-32',
  'state': 'completed',
  'active_cell_executions': 0,
  'pending_notebook_writes': 0,
  'locks_held': 0,
  'notebook_path': '/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-32-breast-cancer/notebooks/kaggle-v020-kaggle-exp-32-breast-cancer.ipynb',
  'reason': None},
 '反復数': 2,
 'CSV_SHA256': '1425d9affa78ba8e53afc81d0ef8a19069ee10c4b21fe89b3cf514071b12ee33',
 '実行順': [22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 34]}